# Chart Quotation & Latest Price

Fetch the SET **intraday chart-quotation** feed for any stock and get the **latest *traded*
price relative to now** — correctly skipping the null-valued minute buckets the API
pre-populates for the rest of the trading day (future minutes, the lunch break, and
no-trade minutes all carry `volume = null`).

Endpoint: `GET /api/set/stock/{symbol}/chart-quotation?period=1D&accumulated=false`

## Setup

Install the library (uncomment if needed), then import the helpers.

In [ ]:
# !pip install settfex

from datetime import datetime

from settfex.services.set import (
    Stock,
    get_chart_quotation,
    get_latest_price,
)

## 1. Fetch the intraday series

`get_chart_quotation` returns a `ChartQuotation` with `prior` (previous close),
`intermissions` (e.g. the lunch break) and a per-minute `quotations` list.

In [ ]:
data = await get_chart_quotation("CPALL", period="1D")

traded = [q for q in data.quotations if q.volume is not None]

print(f"Prior close : {data.prior}")
print(f"Buckets     : {len(data.quotations)} total, {len(traded)} with trades")
print(
    f"Intermission: {data.intermissions[0].begin} -> {data.intermissions[0].end}"
    if data.intermissions
    else "Intermission: none"
)

## 2. The latest *traded* price, right now

The headline capability. `get_latest_price(symbol)` returns the most recent `Quotation`
that actually traded (non-null `volume`) at or before *now* (Asia/Bangkok), or `None` if
nothing has traded yet.

In [ ]:
quotation = await get_latest_price("CPALL")

if quotation is None:
    print("Nothing has traded yet today.")
else:
    print(f"Time   : {quotation.local_datetime}")
    print(f"Price  : {quotation.price}")
    print(f"Volume : {quotation.volume:,.0f}")
    print(f"Change : {quotation.change} ({quotation.percent_change}%)")

### Why not just take the last element?

Because the series spans the **whole session up to end of day** — the tail is usually
null-valued future/no-trade buckets. `get_latest_quotation` scans for the greatest
timestamp `<= as_of` **with a non-null `volume`**, so you always get a real trade.

## 3. Price as of a specific instant (`as_of`)

Pass `as_of` to ask *"what was the latest trade at or before this moment?"* A naive
datetime is treated as **Asia/Bangkok** local time; an aware datetime (any zone) is
converted. Comparisons are always timezone-safe.

In [ ]:
as_of = datetime(2026, 6, 19, 11, 0)  # 11:00, treated as Asia/Bangkok

q = await get_latest_price("CPALL", as_of=as_of)

print(
    "Nothing traded by 11:00"
    if q is None
    else f"Latest trade <= 11:00 -> {q.price} @ {q.local_datetime}"
)

## 4. The `prior` fallback (pre-open)

On the **model**, `get_latest_price(as_of)` returns a scalar price and **falls back to
`prior`** (the previous session's close) when nothing has traded yet — handy for a
"current-ish price" that is never `None` while the market is open.

In [ ]:
data = await get_chart_quotation("CPALL", period="1D")

pre_open = datetime(2026, 6, 19, 9, 0)  # before the 10:00 open

print(f"Scalar price pre-open : {data.get_latest_price(pre_open)}  (falls back to prior)")
print(f"Prior close           : {data.prior}")
print(f"Scalar price now      : {data.get_latest_price()}")

## 5. Using the unified `Stock` class

Every accessor is also available on `Stock`.

In [ ]:
stock = Stock("PTT")

series = await stock.get_chart_quotation(period="1D")

latest = await stock.get_latest_price()

print(f"PTT prior={series.prior}, buckets={len(series.quotations)}")
print(
    "No trades yet" if latest is None else f"PTT latest: {latest.price} @ {latest.local_datetime}"
)

## 6. Hyphenated warrant symbols

Symbol normalization upper-cases and trims but **preserves hyphens**, so warrants like
`JAS-W4` work (lowercase input is fine too).

In [ ]:
q = await get_latest_price("jas-w4")  # normalized to JAS-W4

print("No trades / no data" if q is None else f"JAS-W4 latest: {q.price} @ {q.local_datetime}")

## 7. Build a quick intraday table

Keep only the traded buckets. With the optional `dataframe` extra
(`pip install "settfex[dataframe]"`) you can drop straight into pandas.

In [ ]:
rows = [
    {
        "time": q.local_datetime,
        "price": q.price,
        "volume": q.volume,
        "percent_change": q.percent_change,
    }
    for q in data.quotations
    if q.volume is not None
]

try:
    import pandas as pd

    df = pd.DataFrame(rows)
    display(df.tail())
except ImportError:
    for r in rows[-5:]:
        print(r)

## Summary

- `get_latest_price(symbol)` → the latest **traded** `Quotation` relative to now (or `None`).
- `as_of` lets you ask the same question for any instant; naive = Asia/Bangkok.
- The model's `get_latest_price(as_of)` is a scalar with a `prior` fallback.
- Null future/lunch/no-trade buckets are excluded automatically.

See `docs/settfex/services/set/chart_quotation.md` for the full reference.